# 01 — UQA Data Preparation

This notebook prepares the UQA dataset for Urdu question generation.

## What this notebook does

1. Loads the UQA dataset from Hugging Face
2. Inspects the dataset structure and fields
3. Filters to keep only answerable examples
4. Splits each context into sentences using Urdu delimiters (۔ ؟ !)
5. Locates the sentence containing the answer using character offsets
6. Verifies answer offset/text alignment
7. Marks the answer span with `<ans>` and `</ans>` markers
8. Applies length filtering (source ≤ 60 words, target ≤ 25 words)
9. Saves `train.tsv` and `valid.tsv`
10. Reports statistics and plots length distributions

## Why sentence-level extraction?

The full context paragraph is too long for an efficient seq2seq input.
By extracting only the sentence containing the answer, we give the model
a focused, manageable input that directly contains the information needed
to generate a question.

## Why length filtering?

At this stage the source is still represented as text. We count
whitespace-separated tokens rather than SentencePiece tokens because
the SentencePiece vocabulary has not been trained yet. The length
threshold is a computational filter to remove outlier-length examples
that would waste padding and slow training.

In [ ]:
# Clone repo and install deps (for Colab/Kaggle)
# !git clone https://github.com/YOUR_USERNAME/urdu-qgen-seq2seq.git
# %cd urdu-qgen-seq2seq
# !pip install -r requirements.txt

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from datasets import load_dataset
import csv
import matplotlib.pyplot as plt

from configs.config import DATASET_NAME, DATA_DIR, TRAIN_FILE, VALID_FILE

## Step 1: Load the UQA dataset

In [ ]:
dataset = load_dataset(DATASET_NAME)
print(dataset)
print(f"\nTrain rows: {len(dataset['train'])}")
print(f"Validation rows: {len(dataset['validation'])}")

## Step 2: Inspect the dataset structure

In [ ]:
# Inspect the first few examples
for i in range(3):
    row = dataset['train'][i]
    print(f"--- Example {i} ---")
    print(f"  Question: {row['question']}")
    print(f"  Context:  {row['context'][:100]}...")
    print(f"  Answers:  {row['answers']}")
    print()

## Step 3: Run the full data preparation pipeline

The `prepare_uqa_data()` function handles all processing steps:
sentence splitting, answer offset verification, marker insertion,
length filtering, and TSV output. We call it directly from `src/`.

In [ ]:
from src.data.prepare_data import prepare_uqa_data, prepare_wiki_uqa_data

train_stats, valid_stats = prepare_uqa_data()

In [ ]:
# Also prepare Wiki-UQA for out-of-domain evaluation
wiki_stats = prepare_wiki_uqa_data()

## Step 4: Inspect processed examples

We verify that the `<ans>` markers are correctly placed and the
source/target pairs look reasonable.

In [ ]:
# Show the first 5 examples from the training TSV
with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for i, row in enumerate(reader):
        if i >= 5:
            break
        print(f"--- Example {i+1} ---")
        print(f"  Source: {row['source']}")
        print(f"  Target: {row['target']}")
        print()

## Step 5: Length distribution histograms

These histograms show the whitespace-token counts for source and
target sequences after filtering. They help verify that the length
thresholds (60 source, 25 target) are working correctly.

In [ ]:
from configs.config import FIGURES_DIR
from src.visualization import plot_length_histogram

# Collect lengths from the training TSV
src_lengths, tgt_lengths = [], []
with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        src_lengths.append(len(row['source'].split()))
        tgt_lengths.append(len(row['target'].split()))

plot_length_histogram(
    src_lengths,
    title='Source Sequence Length Distribution (Training)',
    xlabel='Number of whitespace tokens',
    save_path=FIGURES_DIR / 'source_length_histogram.png',
)

plot_length_histogram(
    tgt_lengths,
    title='Target Sequence Length Distribution (Training)',
    xlabel='Number of whitespace tokens',
    save_path=FIGURES_DIR / 'target_length_histogram.png',
)

print(f"Source — min: {min(src_lengths)}, max: {max(src_lengths)}, mean: {sum(src_lengths)/len(src_lengths):.1f}")
print(f"Target — min: {min(tgt_lengths)}, max: {max(tgt_lengths)}, mean: {sum(tgt_lengths)/len(tgt_lengths):.1f}")